In [ ]:
import pandas as pd
import numpy as np
from IPython.display import clear_output

from quick_pp.lithology.multi_mineral import MultiMineral
from quick_pp.plotter.plotter import plotly_log
from quick_pp.qaqc import badhole_flagging, handle_outer_limit

In [ ]:
from quick_pp.plotter.well_log_config import COLOR_DICT

TRACE_DEFS = dict(
    GR=dict(
        track=1,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_width': 1, 'line_color': COLOR_DICT['GR']}
    ),
    RT=dict(
        track=2,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_dash': 'dot', 'line_width': 1, 'line_color': COLOR_DICT['RT']}
    ),
    RHOB=dict(
        track=3,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_width': 1, 'line_color': COLOR_DICT['RHOB']}
    ),
    NPHI=dict(
        track=4,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_width': 1, 'line_color': COLOR_DICT['NPHI']}
    ),
    PEF=dict(
        track=5,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_width': .75, 'line_color': COLOR_DICT['PEF']}
    ),
    DTC=dict(
        track=6,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_width': 1, 'line_color': COLOR_DICT['DTC']}
    ),
    PHIT=dict(
        track=7,
        secondary_y=True,
        hide_xaxis=False,
        style={'line_width': 1, 'line_color': 'blue'}
    ),
    VCLAY=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=False,
        style={
            'line_width': .5, 'line_color': 'black', 'fill': 'tozerox', 'fillpattern_bgcolor': COLOR_DICT['VCLAY'],
            'fillpattern_fgcolor': '#000000', 'fillpattern_fillmode': 'replace', 'fillpattern_shape': '-',
            'fillpattern_size': 2, 'fillpattern_solidity': 0.1, 'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    AVG_ERROR=dict(
        track=9,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_width': 1, 'line_color': 'black'}
    ),
    VFELD=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .5, 'line_color': 'black', 'fill': 'tonextx', 'fillpattern_bgcolor': COLOR_DICT['VFELD'],
            'fillpattern_fgcolor': '#000000', 'fillpattern_fillmode': 'replace', 'fillpattern_shape': '+',
            'fillpattern_size': 3, 'fillpattern_solidity': 0.2, 'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VKAOL=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .5, 'line_color': 'black', 'fill': 'tonextx', 'fillpattern_bgcolor': COLOR_DICT['VKAOL'],
            'fillpattern_fgcolor': '#000000', 'fillpattern_fillmode': 'replace', 'fillpattern_shape': '-',
            'fillpattern_size': 2, 'fillpattern_solidity': 0.1, 'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VCOAL=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .5, 'line_color': 'black', 'fill': 'tonextx', 'fillpattern_bgcolor': COLOR_DICT['VCOAL'],
            'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VANHY=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .5, 'line_color': 'black', 'fill': 'tonextx', 'fillpattern_bgcolor': COLOR_DICT['VANHY'],
            'fillpattern_fgcolor': '#000000', 'fillpattern_fillmode': 'replace', 'fillpattern_shape': 'x',
            'fillpattern_size': 4, 'fillpattern_solidity': 0.1, 'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VPYRI=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .5, 'line_color': 'black', 'fill': 'tonextx', 'fillpattern_bgcolor': COLOR_DICT['VPYRI'],
            'fillpattern_fgcolor': '#000000', 'fillpattern_fillmode': 'replace', 'fillpattern_shape': 'x',
            'fillpattern_size': 2, 'fillpattern_solidity': 0.4, 'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VSILT=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .5, 'line_color': 'black', 'fill': 'tonextx', 'fillpattern_bgcolor': COLOR_DICT['VSILT'],
            'fillpattern_fgcolor': '#000000', 'fillpattern_fillmode': 'replace', 'fillpattern_shape': '.',
            'fillpattern_size': 3, 'fillpattern_solidity': 0.1, 'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VSAND=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .5, 'line_color': 'black', 'fill': 'tonextx', 'fillpattern_bgcolor': COLOR_DICT['VSAND'],
            'fillpattern_fgcolor': '#000000', 'fillpattern_fillmode': 'replace', 'fillpattern_shape': '.',
            'fillpattern_size': 3, 'fillpattern_solidity': 0.1, 'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VCALC=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .5, 'line_color': 'black', 'fill': 'tonextx', 'fillpattern_bgcolor': COLOR_DICT['VCALC'],
            'fillpattern_fgcolor': '#000000', 'fillpattern_fillmode': 'replace', 'fillpattern_shape': '.',
            'fillpattern_size': 3, 'fillpattern_solidity': 0.1, 'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VDOLO=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .5, 'line_color': 'black', 'fill': 'tonextx', 'fillpattern_bgcolor': COLOR_DICT['VDOLO'],
            'fillpattern_fgcolor': '#000000', 'fillpattern_fillmode': 'replace', 'fillpattern_shape': '-',
            'fillpattern_size': 3, 'fillpattern_solidity': 0.3, 'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VGAS=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .1, 'line_color': 'black', 'fill': 'tonextx', 'fillcolor': COLOR_DICT['VGAS'],
            'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VOIL=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .1, 'line_color': 'black', 'fill': 'tonextx', 'fillcolor': COLOR_DICT['VOIL'],
            'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    VWATER=dict(
        track=8,
        secondary_y=False,
        hide_xaxis=True,
        style={
            'line_width': .1, 'line_color': 'black', 'fill': 'tonextx', 'fillcolor': COLOR_DICT['VWATER'],
            'stackgroup': 'litho', 'orientation': 'h'
        }
    ),
    GR_RECONSTRUCTED=dict(
        track=1,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_dash': 'dot', 'line_width': 1, 'line_color': 'black'}
    ),
    RHOB_RECONSTRUCTED=dict(
        track=3,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_dash': 'dot', 'line_width': 1, 'line_color': 'black'}
    ),
    NPHI_RECONSTRUCTED=dict(
        track=4,
        secondary_y=True,
        hide_xaxis=False,
        style={'line_dash': 'dot', 'line_width': 1, 'line_color': 'black'}
    ),
    PEF_RECONSTRUCTED=dict(
        track=5,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_dash': 'dot', 'line_width': 1, 'line_color': 'black'}
    ),
    DTC_RECONSTRUCTED=dict(
        track=6,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_dash': 'dot', 'line_width': 1, 'line_color': 'black'}
    ),
    PHIT_CONSTRUCTED=dict(
        track=7,
        secondary_y=False,
        hide_xaxis=False,
        style={'line_dash': 'dot', 'line_width': 1, 'line_color': 'black'}
    ),
    CALI=dict(
        track=1,
        secondary_y=False,
        hide_xaxis=False,
        style={
            'line_dash': 'dot', 'line_width': 1, 'line_color': COLOR_DICT['CALI'], 'fill': 'tozerox',
            'fillcolor': 'rgba(165, 42, 42, .15)'
        }
    ),
)

In [ ]:
font_size = 8
XAXIS_DEFS = {
    'GR': {
        'title': {'text': 'GR', 'font': {'color': COLOR_DICT['GR'], 'size': font_size}},
        'tickfont': {'color': COLOR_DICT['GR'], 'size': font_size},
        'side': 'top', 'anchor': 'free', 'position': .85,
        'title_standoff': .1, 'dtick': 40, 'range': [0, 200], 'type': 'linear', 'zeroline': False
    },
    'RT': {
        'title': {'text': 'RT', 'font': {'color': COLOR_DICT['RT'], 'size': font_size}},
        'tickfont': {'color': COLOR_DICT['RT'], 'size': font_size},
        'side': 'top', 'anchor': 'free', 'position': .85,
        'title_standoff': .1, 'range': [np.log10(.2), np.log10(2000)], 'type': 'log',
        'tickmode': 'array', 'tickvals': np.geomspace(0.2, 2000, 5), 'tickangle': -90, 'minor_showgrid': True
    },
    'RHOB': {
        'title': {'text': 'RHOB', 'font': {'color': COLOR_DICT['RHOB'], 'size': font_size}},
        'tickformat': ".2f", 'tick0': 1.95, 'dtick': 0.2, 'tickangle': -90,
        'tickfont': {'color': COLOR_DICT['RHOB'], 'size': font_size},
        'side': 'top', 'anchor': 'free', 'position': .85,
        'title_standoff': .1, 'range': [1.95, 2.95], 'type': 'linear'
    },
    'NPHI': {
        'title': {'text': 'NPHI', 'font': {'color': COLOR_DICT['NPHI'], 'size': font_size}},
        'tickfont': {'color': COLOR_DICT['NPHI'], 'size': font_size}, 'zeroline': False,
        'side': 'top', 'anchor': 'free', 'position': .85, 'title_standoff': .1,
        'tickformat': ".2f", 'tick0': -.15, 'dtick': 0.12, 'range': [.45, -.15], 'type': 'linear', 'tickangle': -90
    },
    'VCLAY': {
        'title': {'text': 'LITHOLOGY', 'font': {'color': COLOR_DICT['VSHALE'], 'size': font_size}},
        'tickfont': {'color': COLOR_DICT['VSHALE'], 'size': font_size}, 'range': [0, 1],
        'side': 'top', 'anchor': 'free', 'position': .85, 'title_standoff': .1, 'type': 'linear', 'zeroline': False
    },
    'DTC': {
        'title': {'text': 'DTC', 'font': {'color': COLOR_DICT['DTC'], 'size': font_size}},
        'tickfont': {'color': COLOR_DICT['DTC'], 'size': font_size},
        'side': 'top', 'anchor': 'free', 'position': .85,
        'title_standoff': .1, 'dtick': 20, 'range': [30, 150], 'type': 'linear', 'zeroline': False
    },
    'PEF': {
        'title': {'text': 'PEF', 'font': {'color': COLOR_DICT['PEF'], 'size': font_size}},
        'tickfont': {'color': COLOR_DICT['PEF'], 'size': font_size}, 'zeroline': False,
        'side': 'top', 'anchor': 'free', 'position': .85, 'title_standoff': .1,
        'dtick': 1, 'range': [0, 7], 'type': 'linear', 'showgrid': False
    },
    'GR_RECONSTRUCTED': {
        'title': {'text': 'GR_RECONSTRUCTED', 'font': {'color': 'black', 'size': font_size}},
        'tickfont': {'color': 'black', 'size': font_size},
        'side': 'top', 'anchor': 'free', 'position': .89, 'overlaying': 'x1',
        'title_standoff': .1, 'dtick': 40, 'range': [0, 200], 'type': 'linear', 'zeroline': False
    },
    'RHOB_RECONSTRUCTED': {
        'title': {'text': 'RHOB_RECONSTRUCTED', 'font': {'color': 'black', 'size': font_size}},
        'tickfont': {'color': 'black', 'size': font_size}, 'zeroline': False,
        'side': 'top', 'anchor': 'free', 'position': .89, 'overlaying': 'x3',
        'tickformat': ".2f", 'tick0': 1.95, 'dtick': 0.2, 'tickangle': -90,
        'range': [1.95, 2.95], 'type': 'linear'
    },
    'NPHI_RECONSTRUCTED': {
        'title': {'text': 'NPHI_RECONSTRUCTED', 'font': {'color': 'black', 'size': font_size}},
        'tickfont': {'color': 'black', 'size': font_size}, 'zeroline': False,
        'tickformat': ".2f", 'tick0': -.15, 'dtick': 0.12, 'range': [.45, -.15], 'type': 'linear', 'tickangle': -90,
        'side': 'top', 'anchor': 'free', 'position': .89, 'overlaying': 'x4',
    },
    'PEF_RECONSTRUCTED': {
        'title': {'text': 'PEF_RECONSTRUCTED', 'font': {'color': 'black', 'size': font_size}},
        'tickfont': {'color': 'black', 'size': font_size}, 'zeroline': False,
        'side': 'top', 'anchor': 'free', 'position': .89, 'overlaying': 'x5',
        'dtick': 1, 'range': [0, 7], 'type': 'linear'
    },
    'DTC_RECONSTRUCTED': {
        'title': {'text': 'DTC_RECONSTRUCTED', 'font': {'color': 'black', 'size': font_size}},
        'tickfont': {'color': 'black', 'size': font_size}, 'zeroline': False,
        'side': 'top', 'anchor': 'free', 'position': .89, 'overlaying': 'x6',
        'dtick': 20, 'range': [30, 150], 'type': 'linear'
    },
    'AVG_ERROR': {
        'title': {'text': 'AVG_ERROR', 'font': {'color': 'black', 'size': font_size}},
        'tickfont': {'color': 'black', 'size': font_size}, 'zeroline': False,
        'side': 'top', 'anchor': 'free', 'position': .89, 'type': 'linear', 'range': [0, 15],
    },
    'PHIT': {
        'title': {'text': 'PHIT', 'font': {'color': COLOR_DICT['PHIT'], 'size': font_size}},
        'tickfont': {'color': COLOR_DICT['PHIT'], 'size': font_size},
        'side': 'top', 'anchor': 'free', 'position': .85, 'title_standoff': .1,
        'dtick': 0.1, 'range': [0, 0.5], 'type': 'linear', 'zeroline': False
    },
    'PHIT_CONSTRUCTED': {
        'title': {'text': 'PHIT_CONSTRUCTED', 'font': {'color': 'black', 'size': font_size}},
        'tickfont': {'color': 'black', 'size': font_size},
        'side': 'top', 'anchor': 'free', 'position': .89, 'title_standoff': .1,
        'dtick': 0.1, 'range': [0, 0.5], 'type': 'linear', 'zeroline': False, 'overlaying': 'x7',
    },
    'CALI': {
        'title': {'text': 'CALI', 'font': {'color': COLOR_DICT['CALI'], 'size': font_size}},
        'tickfont': {'color': COLOR_DICT['CALI'], 'size': font_size},
        'side': 'top', 'anchor': 'free', 'position': .89, 'title_standoff': .1, 'overlaying': 'x1',
        'dtick': 6, 'range': [6, 24], 'type': 'linear', 'showgrid': False
    },
}

In [ ]:
%matplotlib inline

from quick_pp.database.objects import Project
from quick_pp.database.db_connector import DBConnector

db_conn = DBConnector()

# Load well from saved file
project_name = "30-7"
well_name = '30-7a-7'
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    df = project.get_all_data()
    well_data = project.get_well_data(well_name)

In [ ]:
# Mask outside threshold
well_data = handle_outer_limit(well_data, True)

neu_den_df = well_data[['DEPTH', 'GR', 'NPHI', 'RHOB', 'DTC', 'PEF']].sort_values('DEPTH').reset_index(drop=True)
minerals = ['CALCITE', 'QUARTZ', 'SHALE', 'DOLOMITE']
df_model = MultiMineral(minerals=minerals).estimate_lithology(
    gr=neu_den_df['GR'], nphi=neu_den_df['NPHI'], rhob=neu_den_df['RHOB'],
    pef=neu_den_df['PEF'], dtc=neu_den_df['DTC']
)
df_model['DEPTH'] = neu_den_df.DEPTH

plot_df = well_data.merge(df_model, how='left', on='DEPTH', suffixes=('_ORI', ''))

In [ ]:
fig = plotly_log(plot_df, well_name=well_name, depth_uom='meter', trace_defs=TRACE_DEFS, xaxis_defs=XAXIS_DEFS)
fig.show(config=dict(scrollZoom=True))
# fig.write_html('plot_no dtc pef carbonate log curves.html')

# Apply to all

In [ ]:
from quick_pp.utils import remove_straights
from quick_pp.porosity import estimate_shale_porosity

final_args = {}
for well_name, well_data in df.groupby('WELL_NAME'):
    # Mask outside threshold
    well_data = handle_outer_limit(well_data, True)

    # Clean up data
    well_data = badhole_flagging(well_data)

    for col in ['GR', 'RT', 'NPHI', 'RHOB']:
        well_data.loc[:, col] = remove_straights(well_data[col])


    neu_den_df = well_data[['DEPTH', 'GR', 'NPHI', 'RHOB', 'DTC', 'PEF']].sort_values('DEPTH').reset_index(drop=True)
    minerals = ['CALCITE', 'QUARTZ', 'SHALE', 'DOLOMITE']
    df_model = MultiMineral(minerals=minerals).estimate_lithology(
        gr=neu_den_df['GR'], nphi=neu_den_df['NPHI'], rhob=neu_den_df['RHOB'],
        pef=neu_den_df['PEF'], dtc=neu_den_df['DTC']
    )
    df_model['DEPTH'] = neu_den_df.DEPTH

    well_data = well_data.merge(df_model, how='left', on='DEPTH', suffixes=('_ORI', ''))

    # Estimate porosity
    phit = well_data.VWATER + well_data.VOIL + well_data.VGAS

    # Calculate vclb: volume of clay bound water and phie
    phit_shale = estimate_shale_porosity(well_data.NPHI, phit)
    vclb = well_data.VCLAY * phit_shale
    well_data['PHIE'] = phit - vclb
    well_data['PHIT'] = phit

    # Save result to database
    with db_conn.get_session() as db_session:
        project = Project(db_session, name=project_name)
        project.update_data(well_data)
        project.save()

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import matplotlib.pyplot as plt

with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    score_df = project.get_all_data()
score_df = score_df[['WELL_NAME', 'CPORE', 'PHIT']].copy()
score_df.dropna(inplace=True)
mape = round(mean_absolute_percentage_error(score_df.CPORE, score_df.PHIT), 2)
r2 = round(r2_score(score_df.CPORE, score_df.PHIT), 2)
print(f"\n ### PHIT MAPE: {mape:.2f}")
print(f" ### PHIT R2: {r2:.2f}")

plt.scatter(score_df.CPORE, score_df.PHIT, label=f'Overall - R2: {r2}, MAPE: {mape}')
for well, data in score_df.groupby('WELL_NAME'):
    mape = round(mean_absolute_percentage_error(data.CPORE, data.PHIT), 2)
    r2 = round(r2_score(data.CPORE, data.PHIT), 2)
    plt.scatter(data.CPORE, data.PHIT, label=f'{well} - R2: {r2}, MAPE: {mape}')
plt.xlabel('Actual')
plt.ylabel('Calculated')
plt.xlim(0, .5)
plt.ylim(0, .5)
plt.legend()